# 声明式编程

声明式语言把过程细节抽象掉,转而关注结果的形式

## 表

一张表也称为关系（relation），它有固定数量的列，每列都有名称和类型。表的每一行代表一条数据记录，每列各有一个值。

## SELECT语句

`SELECT`语句可以定义一张新表:要么列出单行中的值,要么用`from`语句对现有表做投影

In [1]:
# select [column name] from [existing table name]
# where 语句 带过滤条件
# order 语句 结果排序

## 连接

- join语句
- union语句

## 解释SQL

In [2]:
from collections import namedtuple
CitiesRow = namedtuple("Row", ["latitude", "longitude", "name"])

In [3]:
cities = [CitiesRow(38, 122, "Berkeley"),
            CitiesRow(42,  71, "Cambridge"),
            CitiesRow(43,  93, "Minneapolis")]

In [4]:
cities

[Row(latitude=38, longitude=122, name='Berkeley'),
 Row(latitude=42, longitude=71, name='Cambridge'),
 Row(latitude=43, longitude=93, name='Minneapolis')]

In [5]:
DistanceRow = namedtuple("Row", ["name", "distance"])
def select(cities_row):
    latitude, longitude, name = cities_row
    return DistanceRow(name, 60 * abs(latitude-38))
distances = list(map(select, cities))
for row in distances:
    print(row)


Row(name='Berkeley', distance=0)
Row(name='Cambridge', distance=240)
Row(name='Minneapolis', distance=300)


In [25]:
class Select:
    """从 [tables] 中选择 [columns], 满足[conditioin], 按[order]排序"""
    def __init__(self, columns, tables, condition, order):
        self.columns = columns
        self.tables = tables
        self.condition = condition
        self.order = order
        self.make_row = create_make_row(self.columns)
    
    def execute(self, env):
        """把来自各表的行连接、过滤、排序, 再映射到列上"""
        from_rows = join(self.tables, env)
        filtered_rows = filter(self.filter, from_rows)
        ordered_rows = self.sort(filtered_rows)
        return map(self.make_row, ordered_rows)
    
    def filter(self, row):
        if self.condition:
            return eval(self.condition, row)
        else:
            return True
    
    def sort(self, rows):
        if self.order:
            return sorted(rows, key=lambda r: eval(self.order, r))
        else:
            return rows

In [ ]:
def create_make_row(description):
    """
        返回一个从输入环境到一行输出的函数
        description -- 由 [表达式] as [列名] 组成的逗号分割列表
    """
    columns = description.split(", ") # description: name, 60*abs(latitude-38) as distance
    expressions, names = [], []
    for column in columns:
        if " as " in column:
            expression, name = column.split(" as ")
        else:
            expression, name = column, column
        expressions.append(expression)
        names.append(name)
    row = namedtuple("Row", names)
    return lambda env: row(*[eval(e, env) for e in expressions])

In [27]:
from itertools import product
def join(tables, env):
    """
    返回一个迭代器，遍历一行中从名称到值的字典
    tables -- 逗号分割的表名序列
    env    -- 从全局名称到表的字典
    """
    names = tables.split(", ")
    joined_rows = product(*[env[name] for name in names])
    return map(lambda rows: make_env(rows, names), joined_rows)

def make_env(rows, names):
    """创建一个把名称绑定到值的环境"""
    env = dict(zip(names, rows))
    for row in rows:
        for name in row._fields:
            env[name] = getattr(row, name)
    return env

In [32]:
env = {"cities": cities}
select = Select("name, 60*abs(latitude-38) as distance",
                "cities", "name != 'Berkeley'", "-longitude")
# select.condition
for row in select.execute(env):
    print(row)

Row(name='Minneapolis', distance=300)
Row(name='Cambridge', distance=240)


In [35]:
env["distances"] = list(select.execute(env))
joined = Select("cities.name as name, distance, longitude", "cities, distances",
                    "cities.name == distances.name", None)
for row in joined.execute(env):
    print(row)

Row(name='Cambridge', distance=240, longitude=71)
Row(name='Minneapolis', distance=300, longitude=93)


## 递归select语法

In [36]:
# with [tables] select [columns] from [names] where [condition] order by [order]

In [38]:
with states(city, state) as (
        select "Berkeley",  "California"    union
        select "Boston",    "Massachusetts" union
        select "Cambridge", "Massachusetts" union
        select "Chicago",   "Illinois"      union
        select "Pasadena",  "California"
        )
    select a.city, b.city, a.state from states as a, states as b
       where a.state = b.state and a.city < b.city;

SyntaxError: invalid syntax. Perhaps you forgot a comma? (1336607028.py, line 2)